In [1]:
"""
╔══════════════════════════════════════════════════════════════╗
║     Mental Health Dataset Pipeline — v2 (No Leakage)       ║
║     From Data Collection to LLM-Ready Format                ║
╚══════════════════════════════════════════════════════════════╝

Steps:
  1.  Imports & Config
  2.  Download Datasets
  3.  Parse & Unify Format
  4.  ── SPLIT FIRST (before any preprocessing) ──
  5.  Preprocessing Pipeline (fit on train only → transform all)
  6.  Clean & Filter
  7.  EDA
  8.  Emotion Labeling
  9.  LLM Formatting & Save Splits

Key design decisions to prevent data leakage:
  ✅  Split happens BEFORE any statistics-based preprocessing
  ✅  Vocabulary / length stats computed on TRAIN only
  ✅  Deduplication done PER-SPLIT (no cross-split contamination)
  ✅  sklearn Pipeline encapsulates all fit→transform logic
"""

# ════════════════════════════════════════════════════════════════
# STEP 0 — INSTALL
# ════════════════════════════════════════════════════════════════
# !pip install datasets pandas scikit-learn langdetect -q


# ════════════════════════════════════════════════════════════════
# STEP 1 — IMPORTS & CONFIG
# ════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import json
import ast
import re
import warnings
from collections import Counter

from datasets import load_dataset
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# ── Global Config ────────────────────────────────────────────────
CONFIG = {
    # Split ratios (must sum to 1.0)
    "train_ratio": 0.80,
    "val_ratio":   0.10,
    "test_ratio":  0.10,
    "random_seed": 42,

    # Text length bounds (words) — computed/enforced on train, applied to all
    "min_words_user":   3,
    "max_words_user":   300,
    "min_words_assist": 3,
    "max_words_assist": 600,

    # Language filter
    "lang_min_ratio":   0.80,   # fraction of EN/AR chars required

    # LLM format
    "system_prompt": (
        "You are an empathetic mental health support assistant. "
        "Listen carefully and respond with compassion and understanding."
    ),
    "output_prefix": "llm",
}

assert abs(
    CONFIG["train_ratio"] + CONFIG["val_ratio"] + CONFIG["test_ratio"] - 1.0
) < 1e-9, "Split ratios must sum to 1.0"


# ════════════════════════════════════════════════════════════════
# STEP 2 — DOWNLOAD & SAVE DATASETS
# ════════════════════════════════════════════════════════════════
DATASETS_TO_DOWNLOAD = {
    "LuangMV97/Empathetic_counseling_Dataset":    "empathetic",
    "ShenLab/MentalChat16K":                      "mentalchat",
    "CAiRE/prosocial-dialog-zho_Hans":            "prosocial",
    "danlou/safespace-8877-20230920":             "safespace",
    "openchat/cogstack-opengpt-sharegpt":         "cogstack",
    "AllyArc/allyarc_oai_format":                "allyarc",
    "thu-coai/esconv":                            "esconv",
    "Amod/mental_health_counseling_conversations": "amod_counseling",
    "jerryjalapeno/nart-100k-synthetic":          "nart100k",
    "thu-coai/augesc":                            "augesc",
    "mpingale/mental-health-chat-dataset":        "mpingale",
    "chillies/psychology-conversation":           "chillies",
}


def download_all_datasets():
    """Download all datasets from HuggingFace and save as CSV."""
    for hf_path, name in DATASETS_TO_DOWNLOAD.items():
        try:
            ds = load_dataset(hf_path)
            for split in ds.keys():
                fname = f"{name}_{split}.csv"
                ds[split].to_pandas().to_csv(fname, index=False)
                print(f"  ✅ {fname} — {len(ds[split]):,} rows | cols: {ds[split].column_names}")
        except Exception as e:
            print(f"  ❌ {name}: {e}")


# ════════════════════════════════════════════════════════════════
# STEP 3 — HELPER FUNCTIONS & PARSING
# ════════════════════════════════════════════════════════════════
def safe_parse(val):
    """Convert string representation of list/dict to Python object."""
    if isinstance(val, (list, dict)):
        return val
    for parser in (ast.literal_eval, json.loads):
        try:
            return parser(val)
        except Exception:
            pass
    return None


def extract_turns_from_gpt(convs, human_key="human", gpt_key="gpt",
                            role_field="from", val_field="value"):
    """Extract (user, assistant) pairs from sharegpt-style conversations."""
    rows = []
    if not isinstance(convs, list):
        return rows
    for i in range(len(convs) - 1):
        a, b = convs[i], convs[i + 1]
        if a.get(role_field) == human_key and b.get(role_field) == gpt_key:
            rows.append({
                "user_message":      str(a.get(val_field, "")),
                "assistant_message": str(b.get(val_field, "")),
            })
    return rows


def extract_turns_from_oai(msgs):
    """Extract (user, assistant) pairs from OpenAI-style messages."""
    rows = []
    if not isinstance(msgs, list):
        return rows
    for i in range(len(msgs) - 1):
        a, b = msgs[i], msgs[i + 1]
        if a.get("role") == "user" and b.get("role") == "assistant":
            rows.append({
                "user_message":      str(a.get("content", "")),
                "assistant_message": str(b.get("content", "")),
            })
    return rows


def parse_all_datasets() -> pd.DataFrame:
    """
    Parse every dataset into unified format, preserving the original
    'split' label where it exists (train/val/test).
    Returns a single raw DataFrame — splitting is done LATER.
    """
    dfs = []

    # 1. Empathetic Counseling
    for split in ["train", "test"]:
        try:
            df = pd.read_csv(f"empathetic_{split}.csv")
            df = df.rename(columns={"input": "user_message", "label": "assistant_message"})
            df["source"] = "empathetic"
            df["orig_split"] = split
            dfs.append(df[["user_message", "assistant_message", "source", "orig_split"]])
            print(f"  ✅ empathetic [{split}]: {len(df):,}")
        except Exception as e:
            print(f"  ❌ empathetic [{split}]: {e}")

    # 2. MentalChat16K
    try:
        df = pd.read_csv("mentalchat_train.csv")
        df["user_message"]      = df["input"]
        df["assistant_message"] = df["output"]
        df["source"]     = "mentalchat"
        df["orig_split"] = "train"
        dfs.append(df[["user_message", "assistant_message", "source", "orig_split"]])
        print(f"  ✅ mentalchat: {len(df):,}")
    except Exception as e:
        print(f"  ❌ mentalchat: {e}")

    # 3. ProSocial Dialog
    for split in ["train", "validation", "test"]:
        try:
            df = pd.read_csv(f"prosocial_{split}.csv")
            df = df.rename(columns={"context": "user_message", "response": "assistant_message"})
            df["source"]     = "prosocial"
            df["orig_split"] = "val" if split == "validation" else split
            dfs.append(df[["user_message", "assistant_message", "source", "orig_split"]])
            print(f"  ✅ prosocial [{split}]: {len(df):,}")
        except Exception as e:
            print(f"  ❌ prosocial [{split}]: {e}")

    # 4. SafeSpace
    try:
        df   = pd.read_csv("safespace_train.csv")
        rows = []
        for val in df["conversations"]:
            rows.extend(extract_turns_from_gpt(safe_parse(val)))
        ss = pd.DataFrame(rows)
        ss["source"]     = "safespace"
        ss["orig_split"] = "train"
        dfs.append(ss)
        print(f"  ✅ safespace: {len(ss):,}")
    except Exception as e:
        print(f"  ❌ safespace: {e}")

    # 5. CogStack
    try:
        df   = pd.read_csv("cogstack_train.csv")
        rows = []
        for val in df["conversations"]:
            rows.extend(extract_turns_from_gpt(safe_parse(val)))
        cg = pd.DataFrame(rows)
        cg["source"]     = "cogstack"
        cg["orig_split"] = "train"
        dfs.append(cg)
        print(f"  ✅ cogstack: {len(cg):,}")
    except Exception as e:
        print(f"  ❌ cogstack: {e}")

    # 6. AllyArc
    for split in ["train", "test"]:
        try:
            df   = pd.read_csv(f"allyarc_{split}.csv")
            rows = []
            for val in df["messages"]:
                rows.extend(extract_turns_from_oai(safe_parse(val)))
            al = pd.DataFrame(rows)
            al["source"]     = "allyarc"
            al["orig_split"] = split
            dfs.append(al)
            print(f"  ✅ allyarc [{split}]: {len(al):,}")
        except Exception as e:
            print(f"  ❌ allyarc [{split}]: {e}")

    # 7. ESConv
    for split in ["train", "validation", "test"]:
        try:
            df   = pd.read_csv(f"esconv_{split}.csv")
            rows = []
            for val in df["text"]:
                try:
                    entry  = json.loads(val)
                    dialog = entry.get("dialog", [])
                    for i in range(len(dialog) - 1):
                        a, b = dialog[i], dialog[i + 1]
                        if a.get("speaker") == "usr" and b.get("speaker") == "sys":
                            rows.append({
                                "user_message":      a["text"],
                                "assistant_message": b["text"],
                            })
                except Exception:
                    continue
            es = pd.DataFrame(rows)
            es["source"]     = "esconv"
            es["orig_split"] = "val" if split == "validation" else split
            dfs.append(es)
            print(f"  ✅ esconv [{split}]: {len(es):,}")
        except Exception as e:
            print(f"  ❌ esconv [{split}]: {e}")

    # 8. Amod Counseling
    try:
        df = pd.read_csv("amod_counseling_train.csv")
        df = df.rename(columns={"Context": "user_message", "Response": "assistant_message"})
        df["source"]     = "amod_counseling"
        df["orig_split"] = "train"
        dfs.append(df[["user_message", "assistant_message", "source", "orig_split"]])
        print(f"  ✅ amod_counseling: {len(df):,}")
    except Exception as e:
        print(f"  ❌ amod_counseling: {e}")

    # 9. NART 100K
    try:
        df   = pd.read_csv("nart100k_train.csv")
        rows = []
        for val in df["conversations"]:
            rows.extend(extract_turns_from_gpt(safe_parse(val)))
        nt = pd.DataFrame(rows)
        nt["source"]     = "nart100k"
        nt["orig_split"] = "train"
        dfs.append(nt)
        print(f"  ✅ nart100k: {len(nt):,}")
    except Exception as e:
        print(f"  ❌ nart100k: {e}")

    # 10. AugESC
    try:
        df   = pd.read_csv("augesc_train.csv")
        rows = []
        for val in df["text"]:
            try:
                dialog = json.loads(val)
                for i in range(len(dialog) - 1):
                    a, b = dialog[i], dialog[i + 1]
                    if isinstance(a, list) and a[0] == "usr" and b[0] == "sys":
                        rows.append({"user_message": a[1], "assistant_message": b[1]})
            except Exception:
                continue
        ag = pd.DataFrame(rows)
        ag["source"]     = "augesc"
        ag["orig_split"] = "train"
        dfs.append(ag)
        print(f"  ✅ augesc: {len(ag):,}")
    except Exception as e:
        print(f"  ❌ augesc: {e}")

    # 11. Mpingale
    try:
        df = pd.read_csv("mpingale_train.csv")
        df = df.rename(columns={"questionText": "user_message", "answerText": "assistant_message"})
        df["source"]     = "mpingale"
        df["orig_split"] = "train"
        dfs.append(df[["user_message", "assistant_message", "source", "orig_split"]])
        print(f"  ✅ mpingale: {len(df):,}")
    except Exception as e:
        print(f"  ❌ mpingale: {e}")

    # 12. Chillies
    for split in ["train", "validation", "test"]:
        try:
            df = pd.read_csv(f"chillies_{split}.csv")
            df = df.rename(columns={"question": "user_message", "answer": "assistant_message"})
            df["source"]     = "chillies"
            df["orig_split"] = "val" if split == "validation" else split
            dfs.append(df[["user_message", "assistant_message", "source", "orig_split"]])
            print(f"  ✅ chillies [{split}]: {len(df):,}")
        except Exception as e:
            print(f"  ❌ chillies [{split}]: {e}")

    raw = pd.concat(dfs, ignore_index=True)
    print(f"\n  Raw total: {len(raw):,} rows")
    return raw


# ════════════════════════════════════════════════════════════════
# STEP 4 — SPLIT FIRST (no leakage)
# ════════════════════════════════════════════════════════════════
def split_data(df: pd.DataFrame, cfg: dict) -> dict[str, pd.DataFrame]:
    """
    Split strategy:
      • Rows that originally have a test split  → always go to test
      • Rows that originally have a val split   → always go to val
      • Remaining rows are split 80/10/10 (stratified by source)
        using train_test_split twice.

    This respects dataset authors' intended splits and ensures
    no sample appears in more than one set.
    """
    seed = cfg["random_seed"]

    # Rows with explicit test/val labels from original datasets
    test_fixed = df[df["orig_split"] == "test"].copy()
    val_fixed  = df[df["orig_split"] == "val"].copy()
    train_pool = df[df["orig_split"] == "train"].copy()

    # Stratified split of the train pool into train / val_extra / test_extra
    # so the final val/test sizes are always ~10% of total
    val_ratio_adj  = cfg["val_ratio"]  / (cfg["train_ratio"] + cfg["val_ratio"])

    train_df, val_extra = train_test_split(
        train_pool,
        test_size=val_ratio_adj,
        random_state=seed,
        stratify=train_pool["source"],
    )
    # We keep a small extra test pool from the val_extra
    test_ratio_adj = cfg["test_ratio"] / (cfg["val_ratio"] + cfg["test_ratio"])
    val_extra, test_extra = train_test_split(
        val_extra,
        test_size=test_ratio_adj,
        random_state=seed,
        stratify=val_extra["source"],
    )

    splits = {
        "train": train_df,
        "val":   pd.concat([val_fixed,  val_extra],  ignore_index=True),
        "test":  pd.concat([test_fixed, test_extra], ignore_index=True),
    }

    for name, sdf in splits.items():
        splits[name] = sdf.drop(columns=["orig_split"], errors="ignore").reset_index(drop=True)
        print(f"  Split [{name}]: {len(splits[name]):,}")

    # ── Sanity check: no overlap between splits ──────────────────
    key_cols = ["user_message", "assistant_message"]
    train_keys = set(splits["train"][key_cols].apply(tuple, axis=1))
    val_keys   = set(splits["val"][key_cols].apply(tuple, axis=1))
    test_keys  = set(splits["test"][key_cols].apply(tuple, axis=1))

    tv_overlap = len(train_keys & val_keys)
    tt_overlap = len(train_keys & test_keys)
    vt_overlap = len(val_keys   & test_keys)

    print(f"\n  🔒 Leakage check:")
    print(f"     train ∩ val  = {tv_overlap}")
    print(f"     train ∩ test = {tt_overlap}")
    print(f"     val   ∩ test = {vt_overlap}")
    if tv_overlap + tt_overlap + vt_overlap > 0:
        print("  ⚠️  WARNING: overlapping rows detected — removing from val/test")
        splits["val"]  = splits["val"][~splits["val"][key_cols].apply(tuple, axis=1).isin(train_keys)]
        splits["test"] = splits["test"][~splits["test"][key_cols].apply(tuple, axis=1).isin(train_keys)]

    return splits


# ════════════════════════════════════════════════════════════════
# STEP 5 — SKLEARN PREPROCESSING PIPELINE
#            Fit on TRAIN only → transform all splits
# ════════════════════════════════════════════════════════════════

class BasicCleaner(BaseEstimator, TransformerMixin):
    """
    Stateless transforms (safe to run before fitting):
      - Drop null rows
      - Cast to str and strip whitespace
      - Normalize unicode whitespace
      - Remove HTML tags
      - Collapse repeated whitespace
    """
    _HTML_RE  = re.compile(r"<[^>]+>")
    _SPACE_RE = re.compile(r"\s+")

    def fit(self, X, y=None):
        return self  # stateless

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        df = X.copy()
        df = df.dropna(subset=["user_message", "assistant_message"])

        for col in ["user_message", "assistant_message"]:
            s = df[col].astype(str)
            s = s.str.strip()
            s = s.apply(lambda t: self._HTML_RE.sub(" ", t))
            s = s.apply(lambda t: self._SPACE_RE.sub(" ", t).strip())
            df[col] = s

        return df.reset_index(drop=True)


class LanguageFilter(BaseEstimator, TransformerMixin):
    """
    Stateless — keep only rows where 80%+ of user_message chars
    are ASCII (English) or Arabic Unicode block.
    """
    def __init__(self, min_ratio: float = 0.80):
        self.min_ratio = min_ratio

    def fit(self, X, y=None):
        return self

    @staticmethod
    def _is_valid(text: str, min_ratio: float) -> bool:
        text = str(text)
        if not text.strip():
            return False
        good = sum(c.isascii() or "\u0600" <= c <= "\u06FF" for c in text)
        return (good / len(text)) >= min_ratio

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        df   = X.copy()
        mask = df["user_message"].apply(self._is_valid, min_ratio=self.min_ratio)
        return df[mask].reset_index(drop=True)


class WordCountFilter(BaseEstimator, TransformerMixin):
    """
    Fit: compute 1st/99th percentile word counts from TRAIN data
         (only if dynamic_bounds=True).
    Transform: drop rows outside [min, max] word counts.

    With dynamic_bounds=False (default) uses hard config values
    and is fully stateless.
    """
    def __init__(self,
                 min_user: int = 3,   max_user: int  = 300,
                 min_asst: int = 3,   max_asst: int  = 600,
                 dynamic_bounds: bool = False):
        self.min_user  = min_user
        self.max_user  = max_user
        self.min_asst  = min_asst
        self.max_asst  = max_asst
        self.dynamic_bounds = dynamic_bounds

    def fit(self, X: pd.DataFrame, y=None):
        if self.dynamic_bounds:
            uw = X["user_message"].str.split().str.len()
            aw = X["assistant_message"].str.split().str.len()
            # Use 1st–99th percentile of TRAIN data as bounds
            self.min_user_ = int(np.percentile(uw.dropna(), 1))
            self.max_user_ = int(np.percentile(uw.dropna(), 99))
            self.min_asst_ = int(np.percentile(aw.dropna(), 1))
            self.max_asst_ = int(np.percentile(aw.dropna(), 99))
            print(f"  [WordCountFilter] dynamic bounds fitted on train:")
            print(f"    user  words: [{self.min_user_}, {self.max_user_}]")
            print(f"    assist words: [{self.min_asst_}, {self.max_asst_}]")
        else:
            self.min_user_ = self.min_user
            self.max_user_ = self.max_user
            self.min_asst_ = self.min_asst
            self.max_asst_ = self.max_asst
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        df = X.copy()
        uw = df["user_message"].str.split().str.len()
        aw = df["assistant_message"].str.split().str.len()
        mask = (
            (uw >= self.min_user_) & (uw <= self.max_user_) &
            (aw >= self.min_asst_) & (aw <= self.max_asst_)
        )
        return df[mask].reset_index(drop=True)


class TextNormalizer(BaseEstimator, TransformerMixin):
    """
    Stateless text normalization:
      - Lowercase option
      - Remove special characters (keep alphanumeric + basic punctuation)
      - Trim excess punctuation repetitions  (e.g. "!!!" → "!")
    """
    def __init__(self, lowercase: bool = False,
                 remove_special: bool = True,
                 trim_punct: bool = True):
        self.lowercase      = lowercase
        self.remove_special = remove_special
        self.trim_punct     = trim_punct

    _SPECIAL_RE = re.compile(r"[^\w\s.,!?'\-–—\"()/:;]", re.UNICODE)
    _PUNCT_RE   = re.compile(r"([!?.]){2,}")

    def fit(self, X, y=None):
        return self

    def _normalize(self, text: str) -> str:
        if self.lowercase:
            text = text.lower()
        if self.remove_special:
            text = self._SPECIAL_RE.sub(" ", text)
        if self.trim_punct:
            text = self._PUNCT_RE.sub(r"\1", text)
        return re.sub(r"\s+", " ", text).strip()

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        df = X.copy()
        for col in ["user_message", "assistant_message"]:
            df[col] = df[col].apply(self._normalize)
        return df


class DeduplicatorPerSplit(BaseEstimator, TransformerMixin):
    """
    Drop exact (user_message, assistant_message) duplicates
    WITHIN the current split only — no cross-split contamination.
    This is stateless: each split is deduped independently.
    """
    def fit(self, X, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        before = len(X)
        df     = X.drop_duplicates(
            subset=["user_message", "assistant_message"]
        ).reset_index(drop=True)
        print(f"  [Dedup] {before:,} → {len(df):,}  (removed {before - len(df):,})")
        return df


def build_preprocessing_pipeline(cfg: dict) -> Pipeline:
    """
    Assemble the sklearn Pipeline.
    fit() is called ONLY on train split.
    transform() is applied to all splits using the same fitted state.
    """
    return Pipeline([
        ("cleaner",    BasicCleaner()),
        ("lang_filter", LanguageFilter(min_ratio=cfg["lang_min_ratio"])),
        ("wc_filter",  WordCountFilter(
            min_user=cfg["min_words_user"],
            max_user=cfg["max_words_user"],
            min_asst=cfg["min_words_assist"],
            max_asst=cfg["max_words_assist"],
            dynamic_bounds=False,       # Set True to use percentile-based bounds
        )),
        ("normalizer", TextNormalizer(
            lowercase=False,            # keep original case for LLM training
            remove_special=True,
            trim_punct=True,
        )),
        ("dedup",      DeduplicatorPerSplit()),
    ])


def preprocess_splits(splits: dict[str, pd.DataFrame],
                      cfg: dict) -> dict[str, pd.DataFrame]:
    """
    1. Build pipeline
    2. Fit on TRAIN only
    3. Transform all splits with the SAME fitted pipeline
    """
    pipe = build_preprocessing_pipeline(cfg)

    print("\n  ── Fitting pipeline on TRAIN split ──")
    train_clean = pipe.fit_transform(splits["train"])
    print(f"  Train after preprocessing: {len(train_clean):,}")

    processed = {"train": train_clean}

    for split_name in ["val", "test"]:
        print(f"\n  ── Transforming [{split_name}] split ──")
        processed[split_name] = pipe.transform(splits[split_name])
        print(f"  {split_name} after preprocessing: {len(processed[split_name]):,}")

    return processed


# ════════════════════════════════════════════════════════════════
# STEP 6 — CROSS-SPLIT DEDUPLICATION (final leakage check)
# ════════════════════════════════════════════════════════════════
def remove_cross_split_leakage(splits: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    """
    After per-split deduplication, ensure val/test rows do NOT
    appear in the train set.  Train is the ground truth; we remove
    from val/test, never from train.
    """
    key_cols  = ["user_message", "assistant_message"]
    train_set = set(splits["train"][key_cols].apply(tuple, axis=1))

    for name in ["val", "test"]:
        before  = len(splits[name])
        df      = splits[name]
        mask    = ~df[key_cols].apply(tuple, axis=1).isin(train_set)
        splits[name] = df[mask].reset_index(drop=True)
        removed = before - len(splits[name])
        print(f"  [{name}] cross-split leakage removed: {removed:,} rows")

    # Also remove test rows that appear in val
    val_set = set(splits["val"][key_cols].apply(tuple, axis=1))
    before  = len(splits["test"])
    mask    = ~splits["test"][key_cols].apply(tuple, axis=1).isin(val_set)
    splits["test"] = splits["test"][mask].reset_index(drop=True)
    print(f"  [test] val leakage removed: {before - len(splits['test']):,} rows")

    return splits


# ════════════════════════════════════════════════════════════════
# STEP 7 — EDA
# ════════════════════════════════════════════════════════════════
EMOTION_KEYWORDS = {
    "anxiety":    ["anxious", "anxiety", "worried", "worry", "nervous",
                   "panic", "fear", "scared", "stress", "stressed", "overwhelmed"],
    "sadness":    ["sad", "sadness", "depressed", "depression", "grief",
                   "cry", "crying", "hopeless", "miserable", "heartbroken"],
    "anger":      ["angry", "anger", "furious", "rage", "frustrated",
                   "frustration", "mad", "irritated", "annoyed"],
    "happiness":  ["happy", "happiness", "joy", "joyful", "excited",
                   "grateful", "thankful", "hopeful", "proud"],
    "trauma":     ["trauma", "traumatic", "abuse", "abused", "ptsd",
                   "nightmare", "flashback", "assault"],
    "suicide":    ["suicide", "suicidal", "self-harm", "kill myself",
                   "end my life", "worthless", "no reason to live"],
    "loneliness": ["lonely", "loneliness", "alone", "isolated",
                   "isolation", "abandoned", "empty"],
    "confusion":  ["confused", "confusion", "lost", "unsure",
                   "uncertain", "don't know what"],
}


def label_emotion(text: str) -> str:
    """Rule-based emotion detection (priority order matters)."""
    text = str(text).lower()
    for emotion, keywords in EMOTION_KEYWORDS.items():
        if any(kw in text for kw in keywords):
            return emotion
    return "neutral"


def run_eda(splits: dict[str, pd.DataFrame]):
    """Print EDA report across all splits."""
    combined = pd.concat(
        [df.assign(split=name) for name, df in splits.items()],
        ignore_index=True,
    )
    combined["emotion"] = combined["user_message"].apply(label_emotion)
    combined["_uw"]     = combined["user_message"].str.split().str.len()
    combined["_aw"]     = combined["assistant_message"].str.split().str.len()

    print("\n" + "═" * 60)
    print("  EDA REPORT")
    print("═" * 60)
    print(f"  Total rows       : {len(combined):,}")
    print(f"  Nulls            : {combined.isnull().sum().sum()}")
    print(f"  Avg user words   : {combined['_uw'].mean():.1f}  |  median: {combined['_uw'].median():.0f}")
    print(f"  Avg assist words : {combined['_aw'].mean():.1f}  |  median: {combined['_aw'].median():.0f}")

    print(f"\n  Splits:")
    for sp, cnt in combined["split"].value_counts().items():
        print(f"    {sp:10s}: {cnt:,}")

    print(f"\n  Sources (top 10):")
    for src, cnt in combined["source"].value_counts().head(10).items():
        pct = cnt / len(combined) * 100
        print(f"    {src:22s}: {cnt:7,}  ({pct:.1f}%)")

    print(f"\n  Emotions:")
    for em, cnt in combined["emotion"].value_counts().items():
        pct = cnt / len(combined) * 100
        bar = "█" * int(pct / 2)
        print(f"    {em:15s}: {cnt:7,}  ({pct:5.1f}%)  {bar}")

    print(f"\n  User msg length buckets (words):")
    bins   = [0, 10, 20, 50, 100, 200, 99999]
    labels = ["0-10", "11-20", "21-50", "51-100", "101-200", "200+"]
    combined["_bucket"] = pd.cut(combined["_uw"], bins=bins, labels=labels)
    for b, cnt in combined["_bucket"].value_counts().sort_index().items():
        pct = cnt / len(combined) * 100
        print(f"    {str(b):10s}: {cnt:7,}  ({pct:.1f}%)")

    print("═" * 60 + "\n")
    return combined["emotion"]  # return for attaching to splits


# ════════════════════════════════════════════════════════════════
# STEP 8 — LLM FORMATTING & SAVE
# ════════════════════════════════════════════════════════════════
def format_chatml(row: pd.Series, system_prompt: str) -> str:
    """Build a single ChatML-formatted string."""
    return (
        f"<|system|>\n{system_prompt}\n"
        f"<|user|>\n{row['user_message']}\n"
        f"<|assistant|>\n{row['assistant_message']}"
    )


def format_and_save(splits: dict[str, pd.DataFrame], cfg: dict):
    """
    Attach emotion labels, build ChatML text column,
    and save train/val/test CSVs + a merged full CSV.
    """
    system_prompt = cfg["system_prompt"]
    prefix        = cfg["output_prefix"]
    all_dfs       = []

    for split_name, df in splits.items():
        df = df.copy()
        df["emotion"] = df["user_message"].apply(label_emotion)
        df["text"]    = df.apply(
            lambda r: format_chatml(r, system_prompt), axis=1
        )
        df["split"]   = split_name

        path = f"{prefix}_{split_name}.csv"
        df[["text", "source", "emotion", "split"]].to_csv(path, index=False)
        print(f"  💾 {path}: {len(df):,} rows")
        all_dfs.append(df)

    merged = pd.concat(all_dfs, ignore_index=True)
    merged.to_csv("merged_mental_health_clean.csv", index=False)
    print(f"  💾 merged_mental_health_clean.csv: {len(merged):,} rows")
    return merged


# ════════════════════════════════════════════════════════════════
# MAIN — FULL PIPELINE
# ════════════════════════════════════════════════════════════════
def run_pipeline(cfg: dict = CONFIG):
    print("\n╔" + "═" * 57 + "╗")
    print("║      MENTAL HEALTH PIPELINE v2 — START               ║")
    print("╚" + "═" * 57 + "╝\n")

    # ── 1. Download ──────────────────────────────────────────────
    print("► STEP 2 — Download Datasets")
    download_all_datasets()

    # ── 2. Parse → single raw df ─────────────────────────────────
    print("\n► STEP 3 — Parse & Unify (raw, unsplit)")
    raw_df = parse_all_datasets()

    # ── 3. SPLIT FIRST ───────────────────────────────────────────
    print("\n► STEP 4 — Split Data (before any preprocessing)")
    splits = split_data(raw_df, cfg)

    # ── 4. Preprocessing pipeline (fit train → transform all) ────
    print("\n► STEP 5 — Preprocessing Pipeline (no leakage)")
    splits = preprocess_splits(splits, cfg)

    # ── 5. Final cross-split leakage removal ─────────────────────
    print("\n► STEP 5b — Cross-Split Leakage Removal")
    splits = remove_cross_split_leakage(splits)

    # ── 6. EDA ───────────────────────────────────────────────────
    print("\n► STEP 7 — EDA")
    run_eda(splits)

    # ── 7. Format & Save ─────────────────────────────────────────
    print("► STEP 8 — Format & Save")
    merged = format_and_save(splits, cfg)

    total = sum(len(v) for v in splits.values())
    print("\n╔" + "═" * 57 + "╗")
    print("║      PIPELINE COMPLETE ✅                             ║")
    print(f"║      Final rows  : {total:,}")
    print(f"║        train     : {len(splits['train']):,}")
    print(f"║        val       : {len(splits['val']):,}")
    print(f"║        test      : {len(splits['test']):,}")
    print("╚" + "═" * 57 + "╝\n")
    return splits, merged


# ════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    splits, merged = run_pipeline()


╔═════════════════════════════════════════════════════════╗
║      MENTAL HEALTH PIPELINE v2 — START               ║
╚═════════════════════════════════════════════════════════╝

► STEP 2 — Download Datasets


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/8.26M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30937 [00:00<?, ? examples/s]

...

psychology_test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/330945 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8710 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8710 [00:00<?, ? examples/s]

  ✅ chillies_train.csv — 330,945 rows | cols: ['question', 'answer']
  ✅ chillies_validation.csv — 8,710 rows | cols: ['question', 'answer']
  ✅ chillies_test.csv — 8,710 rows | cols: ['question', 'answer']

► STEP 3 — Parse & Unify (raw, unsplit)
  ✅ empathetic [train]: 30,937
  ✅ empathetic [test]: 7,736
  ✅ mentalchat: 16,084
  ✅ prosocial [train]: 120,236
  ✅ prosocial [validation]: 20,416
  ✅ prosocial [test]: 25,029
  ✅ safespace: 0
  ✅ cogstack: 0
  ✅ allyarc [train]: 0
  ✅ allyarc [test]: 0
  ✅ esconv [train]: 10,191
  ✅ esconv [validation]: 2,156
  ✅ esconv [test]: 2,295
  ✅ amod_counseling: 3,512
  ✅ nart100k: 0
  ✅ augesc: 797,783
  ✅ mpingale: 2,775
  ✅ chillies [train]: 330,945
  ✅ chillies [validation]: 8,710
  ✅ chillies [test]: 8,710

  Raw total: 1,387,515 rows

► STEP 4 — Split Data (before any preprocessing)
  Split [train]: 1,166,633
  Split [val]: 104,197
  Split [test]: 116,685

  🔒 Leakage check:
     train ∩ val  = 707
     train ∩ test = 1489
     val   ∩ test 

Mounted at /content/drive
